# Lab: Nonlinearities
## CMSE 381 - Spring 2022
## March 25, 2022



In this module we are going to test out the nonlinear methods we discussed in class from Chapter 7.

In [ ]:
# Everyone's favorite standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import time


# ML imports we've used previously
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import statsmodels.api as sm


# Loading in the data

We're going to use the `Wage` data used in the book, so note that many of your plots can be checked by looking at figures in the book.

In [ ]:
df = pd.read_csv('Wage.csv', index_col =0 )
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

Here's the plot we used multiple times in class to look at a single variable:  `age` vs `wage`

In [ ]:
plt.scatter(df.age,df.wage)
plt.xlabel('Age')
plt.ylabel('Wage')

&#9989; **<font color=red>Do this:</font>** Modify the plot above so that the people earning above 250 are in a different color and/or symbol set.




In [ ]:
# Your code here 

# 1a. Polynomial Regression 

Our first step is to build a polynomial regression model using the age data to predict wage.  So, as in class, we are in $p=1$ world here where we are going to fit the model
$$
\texttt{wage} = \beta_0 + \beta_1 \texttt{age} + \beta_2 \texttt{age}^2 + \cdots + \beta_p \texttt{age}^p +\varepsilon.
$$

The trick here is to build a matrix $X$ which has a column containing `age`, one with `age^2`, one with `age^3`, etc.  Then we hand this to your favorite regression tool (it doesn't need to know it's getting polynomial matrix inputs, it just sees a matrix of features and does it's thing). 

So, here's some code to take our $\texttt{age}$ data column and create a bunch of new columns in our data frame that are simply each the $k$th power of the `age` column

In [ ]:
# Here's the column I care about
df.age

In [ ]:
# Here's one way to get out the pandas series that squares
# each entry
df.age.apply(lambda x: x**2)

&#9989; **<font color=red>Do this:</font>** Use the code above (or any other tricks you might know) to generate a data frame called `polydf` with 4 columns, where the $k$th column has $\texttt{age}^k$




In [ ]:
# Your code here #

Did I need to make you do that? It turns out no. As with many things we've talked about in class, there is already some automated code for us to work with.  In this case, the only difference is that it will hand us back a matrix rather than a data frame. 

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
poly = PolynomialFeatures(4)
X = poly.fit_transform(df.age.values.reshape(-1,1)) #<--- this nastyness is because it wants to be handed a matrix
X[:10,:]

&#9989; **<font color=red>Q:</font>** What other major difference do you notice between the dataframe you constructed above and the matrix provided here? Why do you think that is happening?

*Your answer here*





In [ ]:
# Your code here #

&#9989; **<font color=red>Do this:</font>** Train a linear regression model on these features. What are the coefficients? 


In [ ]:
# Your code here #

&#9989; **<font color=red>Q:</font>** What is the equation for the polynomial that you learned? 

*Your equation here*

&#9989; **<font color=red>Do this:</font>** Draw the polynomial that you learned on top of the age vs wage plot (basically, reconstruct the left side of Fig 7.1 of the textbook, without dealing with the confidence intervals). Note that you can either do this using the polynomial you just figured out, or by using the model you just set up to predict the values. In either case, use the vector of ages `t` below.

In [ ]:
# Your code here #
t = np.linspace(10,100,100)

## Classification version 

Now we can try out the classification version of the problem. Let's build the classifier that predicts whether a person of a given age will make more than $250,000. You already made the matrix of polynomial features, so we just have to hand it to `LogisticRegression` to do its thing.

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
y = np.array(df.wage>250) #<--- this makes sure I just have true/false input
clf = LogisticRegression(random_state=48824)
clf.fit(X,y)

&#9989; **<font color=red>Do this:</font>** Use the model to predict whether or not a person with age given by each entry in your `t` array from above makes above $250K.  Does your result make sense given the right side of Fig 7.1 in the book?


In [ ]:
# Your code here #

Now we'll try to actually recreate Fig 7.1.  

*Note: We should be able to do this with `scikit-learn`, however when I do that, I'm not getting the same function shown in the book and I don't know why.  For now, we'll go over to the stats model package to draw until I can understand why the outputs I'm getting aren't the same.*

In [ ]:
# This part fits our model using statsmodel
clf = sm.GLM(y, X, family=sm.families.Binomial(sm.families.links.logit))
res = clf.fit()
pred2 = res.predict(T)  # Pr(wage>250)

# This part plots the probability distribution
plt.plot(t,pred2)

# What is this doing?
plt.scatter(df.age, y/15, s=30, c='grey', marker='|', alpha=0.7)

plt.xlabel('age')
plt.ylabel('Pr(wage>250|age)')

&#9989; **<font color=red>Q:</font>** What is that scatter plot command doing above?

*Your answer here*

# 1b. Step functions

Now let's try to use step functions to learn a model. Like with the polynomial example above, all we're going to do is build a data frame or feature matrix that has the step function values in each column, and then pass that matrix to our favorite linear modeling function. 

First off, it's easy to find the locations for the knots, which are the places where we switch step functions. Here's the pandas `cut` command, which in this case on some toy data, gives me 3 equal-sized bins, where here, equal-sized means that the width of the intervals are all the same.

In [ ]:
fakeData = np.array([1, 7, 3, 5, 4, 6, 3, 3 , 6,2])
cuts, knots = pd.cut(fakeData, 3,retbins=True)
print(cuts)
print(knots)

The `retbins=True` tells the command to return the breakpoints in the bins, which I saved in my output as `knots`. 

We can either see the intervals chosen by looking at the `categories` saved to the cuts, or by looking at the knots list. 

In [ ]:
print(cuts.categories)
print(knots)

I can find out what bin the $i$th entry is mapped to by just checking the cuts list. 

In [ ]:
i = 5
print('Entry is:', fakeData[i])
print('This comes from bin:', cuts[i])

We can also see how many data points ended up in each interval.

In [ ]:
cuts.value_counts()

Once we've got this list of bins, we can build the data frame that keeps track of all the true/false values for whether a data point is in a particular interval by using the dummy variable trick. 

In [ ]:
pd.get_dummies(cuts)

Then, if I want to figure out which bin is assigned for some other matrix of values that I want to test, I can use the `np.digitize` function as follows.

In [ ]:
u = np.linspace(0,10,12)
print(u)
np.digitize(u,knots)

&#9989; **<font color=red>Q:</font>** What do the entries 0 and 4 correspond to in the output above?

*Your answer here* 

&#9989; **<font color=red>Do this:</font>**
- Use the `cut` tool above to create a feature matrix for the `age` data where each column corresponds to a step function using 4 bins. 
- Drop the first bin.... remember we don't need all of our dummy variables, so we'll just use the remaining 3 to predict.
- Pass this matrix to a linear regression model. 
- Using your model, predict the values on the original $t$ matrix from before.
- Plot the predictions on top of the original sampled data.

*Note: If all goes well, you should have a graph that looks like the right side of Fig 7.2*

In [ ]:
# Your code here #

&#9989; **<font color=red>Do this:</font>** If there's time, modify your code to do classification and recreate the plot on the right side of Fig 7.2.

In [ ]:
# Your code here #

# 2. Splines 

Next, we'll build some spline models. Let's start by playing with some toy data, making heavy use of the examples provided on the [scikitlearn spline page](https://scikit-learn.org/stable/auto_examples/linear_model/plot_polynomial_interpolation.html).

In [ ]:
# Note: this bit is going to use some packages that are newly 
# provided in sklearn 1.0.  If you're having issues, try uncommenting
# and running the update line below.


# pip install --upgrade scikit-learn

In [ ]:
from sklearn.preprocessing import SplineTransformer
from sklearn.pipeline import make_pipeline

In [ ]:
# This code block is going to make us some nasty fake data 
# to try to find some sort of interpolation. 

def f(x):
    """Function to be approximated by polynomial interpolation."""
    return x * np.sin(x)


# whole range we want to plot
x_plot = np.linspace(-1, 11, 100)
y_plot = f(x_plot)


# Make some data.  Provide a small amount of points to make 
# our polynomials all kinds of wiggly.
X = np.linspace(0, 10, 100)
rng = np.random.RandomState(0)
X = np.sort(rng.choice(X, size=20, replace=False))
y = f(X)

# # create 2D-array versions of these arrays to feed to transformers
# X_train = x_train[:, np.newaxis]
# X = X[:, np.newaxis]

X = X.reshape(-1,1)
y = y.reshape(-1,1)


#====ploting======

# plot function
plt.plot(x_plot, y_plot,label="ground truth")

# plot training points
plt.scatter(X, y, label="training points")

plt.legend()

Let's pretend you never saw that $f$ function I used to build the data, you're just handed these scattered data points and asked to learn a piecewise polynomial that fits it. 

In [ ]:
# plot training points
plt.scatter(X, y, label="training points")

plt.legend()

The `SplineTransformer` sets up our basis functions for us. These are the functions that we are learning coefficients for when we are doing regression.

In [ ]:
ax = plt.gca()
splt = SplineTransformer(n_knots=4, degree=3).fit(X)
ax.plot(x_plot, splt.transform(x_plot.reshape(-1,1)))
ax.legend(ax.lines, [f"spline {n}" for n in range(6)])
ax.set_title("SplineTransformer")

# plot knots of spline
knots = splt.bsplines_[0].t
ax.vlines(knots[3:-3], ymin=0, ymax=0.8, linestyles="dashed")


I'm going to make use of a nice function from scikitlearn that builds up a pipeline for us to use.  Basically, the `make_pipline` function here takes your input data, runs the `SplineTransformer` on it to get the features, then runs `Ridge` regression.  

In [ ]:
# B-spline with 4 + 3 - 1 = 6 basis functions
model = make_pipeline(SplineTransformer(n_knots=4, degree=3), LinearRegression())
model.fit(X, y)


y_hat = model.predict(X)


In [ ]:
spline_y_plot = model.predict(x_plot.reshape(-1,1))


# plot training points
plt.scatter(X, y, label="Training points")

plt.scatter(X,y_hat,label = 'Predicted',marker = '+')

plt.legend()


plt.plot(x_plot,spline_y_plot)

&#9989; **<font color=red>Do this:</font>** 
- Using the code above that generates splines, build a cubic spline model to predict wage from age. 
- Use your trained model to draw the learned spline on the scatter plot of age and wage, as in the left side of Fig 7.5. (*Note we're only doing regular splines with this code, not natural as in Fig 7.4, but the results end up pretty similar in our case*)

In [ ]:
# Your code here #

# 3. GAMs

Our last trick is to build up some generalized linear models (GAMs) by combining our non-linear tools together for different input variables. 

Rather than doing this fully manually (which we could do by aplying our functions to each column and passing the result along to our favorite modeling tool), I'm going to make use of a nice package, `pygam`, which puts these things together for me. 

- [pygam documentation](https://pygam.readthedocs.io)
- [Quick start guide](https://pygam.readthedocs.io/en/latest/notebooks/quick_start.html)

In [ ]:
# Uncomment here to install the pygam package. 
# pip install pygam

from pygam import LinearGAM, s, f

In [ ]:
# I want to predict using three of my input variables
df_gam = df[['year','age','education']]
df_gam.head()

However, pygam inputs require numpy arrays, which require no columns of strings like our `education` data. I'm going to cheat and let pygam do my data cleanup for me. They need a numpy array rather than a dataframe. This version of the $X$ matrix has just converted our education entries into 0 through 3 (Wheee python numbering system!). 

In [ ]:
from pygam.datasets import wage

X, y = wage()

X[:10,:]

This package has several nice built in tricks, that essentially recreate some of the work we did above.  

Namely, the tersely named `s` function builds a spline on the features its handed, and the also tersely named `f` function takes in factors (so does our dummy variable trick). 

In [ ]:
from pygam import LinearGAM, s, f

gam = LinearGAM(s(0,n_splines = 4) + s(1,n_splines = 4) + f(2)).fit(X, y)

In [ ]:
gam.summary()

Welcome to the fun of open source code! They put in a warning that there's a known bug, so the p-values are not to be trusted. The deeper you get in this world, the more interesting these errors become. Then you start contributing to try to fix them, and slowly everything becomes more and more stable. 

In any case, they have some built in code to recreate the plots like those you see in Fig 7.12 where we can see the predicted values and the confidence intervals

In [ ]:
for i, term in enumerate(gam.terms):
    if term.isintercept:
        continue

    XX = gam.generate_X_grid(term=i)
    pdep, confi = gam.partial_dependence(term=i, X=XX, width=0.95)

    plt.figure()
    plt.plot(XX[:, term.feature], pdep)
    plt.plot(XX[:, term.feature], confi, c='r', ls='--')
    plt.title(repr(term))
    plt.show()

&#9989; **<font color=red>Do this:</font>** The function `LogisticGAM` from the `pygam` package does classification. Use this to modify the code above to build a GAM to predict whether a person will have income above $250. See if you can recreate Fig 7.13 and/or 7.14.

*By way of warning, my results aren't giving exactly the same functions as in the figures, but you should be able to get relatively close*

In [ ]:
# Your code here #

# Lab Survey

To get credit for today's lab, fill out the following survey before the end of class:

https://forms.gle/hX8GT5FJ2fNMeTo1A

Note this is the same link for every lab, so you will fill this out multiple times this semester.



-----
### Congratulations, we're done!
Written by Dr. Liz Munch, Michigan State University

<a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc/4.0/88x31.png" /></a><br />This work is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/">Creative Commons Attribution-NonCommercial 4.0 International License</a>.